In [22]:
import os
import sys
import logging
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "config.py").exists():
    BASE_DIR = CURRENT_DIR
else:
    BASE_DIR = CURRENT_DIR / "rpi_system"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# Import configurations from config.py
import config
from config import (
    DATA_DIR,
    MODELS_DIR,
    OUTPUT_TFLITE,
    OUTPUT_LABELS,
    IMAGE_SIZE,
    BATCH_SIZE,
    EPOCHS,
    LEARNING_RATE,
)

import tensorflow as tf
from tensorflow.keras import layers, models

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("Trainer")

print(" Setup complete! TensorFlow version:", tf.__version__)
print(" Dataset path:", DATA_DIR)

 Setup complete! TensorFlow version: 2.20.0
 Dataset path: C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\Data


In [23]:
logger.info(f"TensorFlow version: {tf.__version__}")
logger.info(f"Loading images from: {DATA_DIR}")

if not DATA_DIR.exists():
    logger.error(f"Data directory not found at: {DATA_DIR}")
    sys.exit(1)


18:40:21 [INFO] TensorFlow version: 2.20.0
18:40:21 [INFO] Loading images from: C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\Data


In [24]:
# Load train and validation datasets (80% train, 20% val)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

Found 1394 files belonging to 11 classes.
Using 1116 files for training.
Found 1394 files belonging to 11 classes.
Using 278 files for validation.


In [25]:
# Extract and save class labels
class_names = train_ds.class_names
num_classes = len(class_names)
logger.info(f"Found {num_classes} animal classes: {class_names}")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_LABELS, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(f"{name}\n")
logger.info(f"Saved class labels to: {OUTPUT_LABELS}")

# Optimize dataset loading with prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(100).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

18:40:25 [INFO] Found 11 animal classes: ['Deer', 'Dog', 'Dolphin', 'Elephant', 'Giraffe', 'Horse', 'Kangaroo', 'Lion', 'Panda', 'Tiger', 'Zebra']
18:40:25 [INFO] Saved class labels to: C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\rpi_system\models\labels.txt


In [26]:
# Data Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name="data_augmentation")

In [27]:
# MobileNetV3 small Base (Pre-trained on ImageNet)
preprocess_input = tf.keras.applications.mobilenet_v3.preprocess_input
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # Freeze pre-trained feature extractor

In [28]:
# Build Classification Head
inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="edge_ai_animal_detector")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [29]:
# Train
logger.info(f"Training for {EPOCHS} epochs with EarlyStopping...")
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

val_loss, val_acc = model.evaluate(val_ds)
logger.info(f"Training complete! Final validation accuracy: {val_acc:.1%}")

18:40:32 [INFO] Training for 20 epochs with EarlyStopping...


Epoch 1/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 13s 231ms/step - accuracy: 0.3118 - loss: 2.0581 - val_accuracy: 0.7086 - val_loss: 1.2586 - learning_rate: 0.0010
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 186ms/step - accuracy: 0.6738 - loss: 1.1225 - val_accuracy: 0.8813 - val_loss: 0.6960 - learning_rate: 0.0010
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 188ms/step - accuracy: 0.7858 - loss: 0.7860 - val_accuracy: 0.9065 - val_loss: 0.5124 - learning_rate: 0.0010
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 189ms/step - accuracy: 0.8073 - loss: 0.6667 - val_accuracy: 0.9245 - val_loss: 0.3775 - learning_rate: 0.0010
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 186ms/step - accuracy: 0.8351 - loss: 0.5794 - val_accuracy: 0.9424 - val_loss: 0.3199 - learning_rate: 0.0010
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 190ms/step - accuracy: 0.8737 - loss: 0.5049 - val_accuracy: 0.9317 - val_loss: 0.2941 - learning_rate: 0.0010
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 188ms/step - accuracy: 0.8566 - loss: 0.4899 -

18:42:59 [INFO] Training complete! Final validation accuracy: 97.1%


In [ ]:
# Convert to Quantized TensorFlow Lite for Raspberry Pi
logger.info("Converting model to optimized .tflite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(OUTPUT_TFLITE, "wb") as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
logger.info(f"Saved TFLite model to: {OUTPUT_TFLITE} ({size_mb:.2f} MB)")
logger.info("Ready! Now classifier.py can load this model.")

18:43:08 [INFO] Converting model to optimized .tflite...
18:43:09 [INFO] Function `function` contains input name(s) resource with unsupported characters which will be renamed to edge_ai_animal_detector_1_dense_2_1_biasadd_readvariableop_resource in the SavedModel.
18:43:09 [INFO] Function `function` contains input name(s) resource with unsupported characters which will be renamed to edge_ai_animal_detector_1_dense_2_1_biasadd_readvariableop_resource in the SavedModel.


INFO:tensorflow:Assets written to: C:\Users\sabbu\AppData\Local\Temp\tmp2fm5kmgi\assets


18:43:11 [INFO] Assets written to: C:\Users\sabbu\AppData\Local\Temp\tmp2fm5kmgi\assets


Saved artifact at 'C:\Users\sabbu\AppData\Local\Temp\tmp2fm5kmgi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_549')
Output Type:
  TensorSpec(shape=(None, 11), dtype=tf.float32, name=None)
Captures:
  2534981957200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2534951047760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2534949403728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2534981954896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2534951043536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2535718125392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2534949399312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2535718125200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2534949403344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2534949402768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  

18:43:15 [INFO] Saved TFLite model to: C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\rpi_system\models\animal_classifier.tflite (1.07 MB)
18:43:15 [INFO] Ready! Now classifier.py can load this model.


Testing the saved model

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import config

interpreter = tf.lite.Interpreter(model_path=str(config.OUTPUT_TFLITE))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

test_image_path = str(config.DATA_DIR / "Panda" / "Panda_1_1.jpg")
img = cv2.imread(test_image_path)
rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
resized_img = cv2.resize(rgb_img, (config.INPUT_WIDTH, config.INPUT_HEIGHT))
input_data = np.expand_dims(resized_img, axis=0).astype(np.float32)

input_data = tf.keras.applications.mobilenet_v3.preprocess_input(input_data)

interpreter.set_tensor(input_details[0]["index"], input_data)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]["index"])

with open(config.OUTPUT_LABELS, "r") as f:
    labels = [line.strip() for line in f if line.strip()]

top_idx = int(np.argmax(output[0]))
confidence = float(output[0][top_idx])

print(f" Predicted Animal: {labels[top_idx]}")
print(f" Confidence Score: {confidence * 100:.2f}%")


 Predicted Animal: Panda
 Confidence Score: 99.97%
